<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/09_dynamic_programming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stochastic dynamic programming
*(Adapted from Stanford AA 203)*

In this problem we will explore discrete-time dynamic programming for stochastic systems; that is, systems where the result of taking a certain action is not deterministic, but instead any of a set of results may occur, according to some known probability distribution. In this case, we cannot optimize the value function directly, since even choosing a known sequence of actions will not always give in the same result. Instead, we optimize the [_expected value_](https://en.wikipedia.org/wiki/Expected_value) of the value function instead (if it's been a while since you've taken a probability class, or if you've never taken one, that Wikipedia article may be helpful).






Suppose we have a machine that is either running or is broken down. If it runs throughout one week, it makes a gross profit of \$100. If it fails during the week, gross profit is zero. If it is running at the start of the week and we could perform preventive maintenance, and the probability that it will fail during the week is 0.4. If we do not perform such maintenance, the probability of failure is 0.7. However, maintenance will cost \$20. If the machine is broken down at the start of the week, it may either be repaired at a cost of \$40, in which case it will fail during the week with a probability of 0.4. At any time, we could also replace the machine at a cost of \$150 with a new machine; a new machine is guaranteed to run through its first week of operation. Performing maintenance on a broken machine does not fix it and the machine will remain broken.
Similarly, repairing a running machine does not improve it and it is equivalent to doing nothing (i.e., you call the repair technician to come but they do nothing and you still have to pay them.)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## (a) What are the states?


In [ ]:
state_space = ["state 1", "state 2"]  # UPDATE ME!


## (b) What are the actions?
NOTE: actions are synonymous with controls. Instead of $u$ for controls, we use $a$ for actions.

In [ ]:
action_space = ["action 1", "action 2"]  # UPDATE ME!


## (c) Write out the transition probability matrix T

Suppose we have finite discrete states ${s_1, s_2,\ldots, s_N}$ and actions ${a_1, a_2,\ldots, a_M}$.
Then $T_{ijk}$ denotes the the probability of moving from $s_i$ to $s_j$ when taking action $a_k$.

To (hopefully) make it a little bit easier, we will use a dictionary with keys referring to actions, and the dictionary value refer to the transition matrix corresponding to that action.


In [ ]:
# transition matrices for each action

transition_matrices = {}
# T_{ij} = P(s_next = j | s_current = i, action)
# s_current = running, s_current = broken

# Transition matrix when taking action = nothing
transition_matrices[action_space[0]] = np.array(
    [
        [..., ...],  # s_current = running # change me
        [..., ...],  # s_current = broken # change me
    ]
)

# may add more depending on number of actions



# these functions below are provided for you, you do not need to change it
# it uses the transition matrices defined above

def get_transition_probability(s_current: str, s_next: str, action: str):
    """Returns the transition probability of moving from s_current to s_next when taking action."""

    # get the transition matrix for the action
    action_matrix = transition_matrices[action]

    # get the index of the current state
    s_current_index = state_space.index(s_current)

    # get the index of the next state
    s_next_index = state_space.index(s_next)

    # get the transition probability based on the index of the current and next state
    return action_matrix[s_current_index, s_next_index]

def get_transition_probability_all(s_current: str, action: str):
    """Returns the transition probabilities of moving from s_current to all next states when taking action."""

    # get the transition matrix for the action
    action_matrix = transition_matrices[action]

    # get the index of the current state
    s_current_index = state_space.index(s_current)

    # get the transition probabilities based on the index of the current state
    return action_matrix[s_current_index, :]

def sample_next_state(s_current: str, action: str, k: int = 1):
    """Given a current state and an action, sample the next state based on transition probabilities.
    Arguments:
        s_current: current state
        action: action taken
        k: number of samples to draw
    Returns:
        list of sampled next states
    """
    # get the transition probabilities for the current state and action
    probabilities = get_transition_probability_all(s_current, action)

    # sample the next state based on the transition probabilities
    return [state_space[i] for i in np.random.choice(range(len(state_space)), p=probabilities, size=k).tolist()]

## (d) Write out the state-action-next state reward
Given the states, actions, and transitions, write out the values for $r(s_t, a_t, s_{t+1})$.



In [ ]:
# reward matrices for each action

reward_matrices = {}
# R_{ij} = R(s_current = i, s_next = j, action)
# s_current = running, s_current = broken

# Reward matrix when taking action = nothing
reward_matrices[action_space[0]] = np.array(
    [
        [..., ...],  # s_current = running
        [..., ...],  # s_current = broken
    ]
)

# may add more depending on number of actions


# these function below is provided for you, you do not need to change it
# it uses the reward matrices defined above
def get_reward(s_current, s_next, action):
    """Returns the reward of moving from s_current to s_next when taking action."""

    # get the reward matrix for the action
    reward_matrix = reward_matrices[action]

    # get the index of the current state
    s_current_index = state_space.index(s_current)

    # get the index of the next state
    s_next_index = state_space.index(s_next)

    # get the reward for the current state and next state
    return reward_matrix[s_current_index, s_next_index]

## (e) Run dynamic programming to find optimal policy!
Using dynamic programming, find the optimal repair, replacement, and maintenance policy that maximizes total expected profit over four weeks, assuming a new machine at the start of the first week.




### (i) Set up your value function. What is V_terminal?
We need to set up our value function table that stores all the value for all the states at each time.
Note that the time steps we have are refer to the start of the week.

Week 0 (t=0), Week 1 (t=1), Week 2 (t=2), Week 3 (t=3), Week 4 (t=4)

Recall that we only get a reward for making it *through* a week, but there is no reward for being in a particular state at the start of the week.
That said, determine what the terminal value function is, which is needed to initialize the dynamic programming algorithm.


In [ ]:
V_terminal = np.array([1.0, 1.0])  # UPDATE ME!


# this function below is provided for you, you do not need to change it
def get_value(state, V_table):
    """Given a state and a value function table, return the value of the state."""
    state_index = state_space.index(state)
    return V_table[state_index]


### (ii) Perform the bellman update


In [ ]:
def bellman_update(V_current, discount_factor=1.0):
    """Perform a Bellman update to compute the next value function table.
    Args:
        V_current: current value function table
    Returns:
        V_next: updated value function table

    V_next[s_current] = max over actions of [ sum over s_next of ( reward(s_current, s_next, action) + discount_factor * V_current[s_next] ) ]
    """
    #

    V_next = np.zeros_like(V_current)
    policy = {}
    for s_current in state_space: # iterate over the states
        Qs = np.stack([
            sum(
                (
                    # get the reward for the current state and next state
                    ...
                    # get the value for the next state
                    + ...
                )
                # get the transition probability for the current state and next state
                * ...
                for s_next in state_space
            )
            for action in action_space
        ])
        # get the value for the current state based on the maximum reward
        V_next[state_space.index(s_current)] = np.max(Qs)
        # get the action for the current state based on the maximum reward
        policy[s_current] = action_space[np.argmax(Qs)]
    return V_next, policy


In [ ]:
V_next, policy = bellman_update(V_terminal)  # test the function
V_next, policy

### (iii) Now run the Bellman update over the 4 weeks!

In [ ]:
n_weeks = 4
value_function = {}
value_function[n_weeks] = V_terminal
policy_list = {}


for week in range(n_weeks): # iterate over the weeks
    print(f"Week {n_weeks - week}")
    V_current = value_function[n_weeks - week] # get the value function for the current week
    V_next, policy = bellman_update(V_current) # update the value function for the next week
    value_function[n_weeks - week - 1] = V_next # store the value function for the next week
    policy_list[n_weeks - week - 1] = policy # store the policy for the next week
value_function, policy_list

For n number of trials, simulate the policy and get the empirical mean reward and the expected value from the Bellman equation. These numbers should be equal as `n_trial` goes to infinity.

In [ ]:
# np.random.seed(0) # set seed for reproducibility
start_state = state_space[0] # start in the first state
reward_list = [] # list of rewards
n_trials = 10000

for _ in range(n_trials): # iterate over the trials
    rewards = []
    states_list = [start_state] # list of states
    for week in range(n_weeks): # iterate over the weeks
        state = states_list[-1] # get the last state
        action = policy_list[week][state]
        next_state = sample_next_state(state, action)[0] # sample the next state
        states_list.append(next_state) # add the next state to the list
        reward = get_reward(state, next_state, action) # get the reward for the current state and next state
        rewards.append(reward)
        # print(f"Week {week}: State: {state}, Action: {action}, Next State: {next_state}, Reward: {reward}")


    # convert the rewards to an array to plot the histogram
    rewards = np.array(rewards)
    reward_list.append(np.sum(rewards))
reward_list = np.array(reward_list)

# get the empirical mean reward and the expected value from the Bellman equation
empirical_mean_reward = np.mean(reward_list)
print(f"Mean reward (empirical): {empirical_mean_reward:.2f} \nExpected value (Bellman): {get_value(start_state, value_function[0]):.2f}")

plt.hist(reward_list, bins=30)
plt.grid(alpha=0.3)
plt.xlabel("Total Reward over 4 Weeks")
plt.ylabel("Frequency")
plt.title("Histogram of Total Rewards from Simulations")
plt.show()